# Importing the Packages

In [ ]:
!pip3 install openai chromadb python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

# Load the Packages

In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import chromadb

In [ ]:
from google.colab import userdata
openaiapikey = userdata.get('OPENAI_API_KEY')
print(openaiapikey[:10])

sk-proj-T3


In [ ]:
client = OpenAI(api_key=openaiapikey)
print("OpenAI client created successfully!!")

OpenAI client created successfully!!


# Communicaitng to the OpenAI Api

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},]
)
print(response.choices[0].message.content)

The capital of France is Paris.


In [ ]:
response.model_dump_json()

'{"id":"chatcmpl-E5tdl94dSpEVlWbeIlcUOqF9bJNOy","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"RAG, or Retrieval-Augmented Generation, is a technique in natural language processing that enhances the capabilities of generative models by incorporating external information. Here are three key points about RAG:\\n\\n- **Retrieval Component**: RAG integrates a retrieval mechanism that fetches relevant information from a large knowledge base or document collection based on the input query. This allows the model to access up-to-date and specific information that may not be contained within its training data.\\n\\n- **Generative Component**: After retrieving relevant documents or snippets, RAG utilizes a generative model (like GPT) to synthesize responses or generate text that incorporates both the retrieved data and the original query, enhancing the relevance and accuracy of the output.\\n\\n- **Improved Performance**: By combining retrieval and generation,

# Loading and Chunking the data from the documents

In [ ]:
with open("/content/_data/company_hr_policy.txt", "r") as f:
  hr_document = f.read()

with open("/content/_data/engineering_standards.txt", "r") as f:
  engineering_document = f.read()

with open("/content/_data/onboarding_guide.txt", "r") as f:
  onboarding = f.read()

with open("/content/_data/product_knowledge_base.txt", "r") as f:
  prodcut = f.read()

with open("/content/_data/security_policy.txt", "r") as f:
  security = f.read()
print(engineering_document)

NovaTech Solutions — Engineering Standards & Practices
Version 2.0 | Last Updated: March 2026

SECTION 1: GIT WORKFLOW

Branching Strategy:
We follow a simplified Git Flow. The main branch is always deployable. All development happens in feature branches.

Branch naming convention:
- feature/JIRA-123-short-description (for new features)
- fix/JIRA-456-bug-description (for bug fixes)
- hotfix/JIRA-789-critical-fix (for production emergencies)

Pull Request Rules:
Every change must go through a Pull Request (PR). Direct commits to main are blocked. PRs require at least 1 approval from a team member. The PR author cannot approve their own PR. All CI checks must pass before merging. PRs should be small — ideally under 400 lines of code changes.

Commit Message Format:
Use conventional commits format:
- feat: add user authentication endpoint
- fix: resolve null pointer in payment service
- docs: update API documentation for v2
- refactor: simplify database connection logic
- test: add unit 

In [ ]:
print(f"HR Policy Document : {len(hr_document)}")
print(f"Engineering Document : {len(engineering_document)}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

HR Policy Document : 7464
Engineering Document : 4941
HR Policy Words: 1052
Engineering Words: 680


# Chunking Straregy

In [ ]:
def chunk_document(text, source_name):
    """
    Split a document into chunks by paragraph.
    Returns a list of dicts with text and metadata.
    """
    # Split on double newlines
    paragraphs = text.strip().split("\n\n")

    chunks = []
    for para in paragraphs:
        para = para.strip()
        # Skip short lines (headers, separators)
        if len(para) < 50:
            continue
        # Skip separator lines
        if para.startswith("===="):
            continue
        chunks.append({
            "text": para,
            "source": source_name
        })

    return chunks

In [ ]:
hr_chunks = chunk_document(hr_document, "HR Policy")
engineering_chunks = chunk_document(engineering_document, "Engineering")
onboarding = chunk_document(onboarding, "Onboarding")
product = chunk_document(prodcut, "Product")
security = chunk_document(security, "Security")

In [ ]:
print(len(hr_chunks))
print(len(engineering_chunks))

25
18


In [ ]:
all_chunks = hr_chunks + engineering_chunks + onboarding + product + security
print(len(all_chunks))

120


# Embedding Mnaually?

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = ''' Office Network:
- Connect only to the "NovaTech-Secure" WiFi network (WPA3)
- The "NovaTech-Guest" network is for visitors only — employees should not use it
- Do not connect personal IoT devices (smart speakers, etc.) to the office network '''

embeddings = model.encode(sentences)

print(embeddings.shape)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(384,)
[-2.71274485e-02  3.71070653e-02  2.69963723e-02 -3.76516283e-02
 -1.24880066e-02  3.69489379e-02  2.72595380e-02 -9.89013910e-02
  1.21910665e-02 -5.96426288e-03 -2.40616128e-02  9.83860791e-02
 -1.46336481e-02 -7.14524463e-03 -9.63099301e-03 -3.73132806e-03
  4.77994941e-02 -6.60055727e-02  8.68471265e-02  5.59232198e-02
  6.26666099e-02 -2.16602869e-02  5.59134549e-03 -4.52482216e-02
  1.12407394e-01  1.82983316e-02  1.06359655e-02  8.20038393e-02
 -1.59085710e-02  1.81507692e-02 -2.22688522e-02  1.61230529e-03
  2.39937130e-04  4.42378484e-02 -4.55207564e-02 -1.53136343e-01
 -4.13970463e-03  6.60401583e-02 -2.05189139e-02 -1.09434938e-02
 -1.74337924e-02 -5.66976070e-02 -7.42021725e-02  4.75947671e-02
  9.90143511e-03 -3.26358899e-02 -6.63817152e-02 -5.27318791e-02
  1.69445574e-02  1.43104736e-02  1.61807835e-02 -7.82664195e-02
 -6.00268096e-02  1.23774938e-01  2.48117838e-02 -8.87978598e-02
  8.00592918e-03  2.06095241e-02  3.16744149e-02 -4.53798249e-02
  7.19641596e-02 -

In [ ]:
sentence1 = ''' I love chicken Biriyani '''
sentence2 = ''' I love eating flaovured rice with chicken'''
sentence3 = ''' I am doing work from Home'''

embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)
embedding3 = model.encode(sentence3)

print(embedding1.shape)
print(embedding2.shape)
print(embedding3.shape)

(384,)
(384,)
(384,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity1 = cosine_similarity([embedding1], [embedding2])
similarity2 = cosine_similarity([embedding1], [embedding3])
similarity3 = cosine_similarity([embedding2], [embedding3])

print(similarity1)
print(similarity2)
print(similarity3)

[[0.56787026]]
[[0.01995524]]
[[0.10406426]]


# Storing Chunks with Chroma DB

In [ ]:
chroma_clinet = chromadb.Client()
collection = chroma_clinet.create_collection(name="company_docs")

In [ ]:
documents = []
ids = []
metadatas = []

for i, chunk in enumerate(all_chunks):
  documents.append(chunk['text'])
  ids.append(f"chunk_{i}")
  metadatas.append({"source": chunk["source"]})

print(documents[110])
print(ids[110])
print(metadatas[110])

Office Network:
- Connect only to the "NovaTech-Secure" WiFi network (WPA3)
- The "NovaTech-Guest" network is for visitors only — employees should not use it
- Do not connect personal IoT devices (smart speakers, etc.) to the office network
chunk_110
{'source': 'Security'}


In [ ]:
collection.add(
    documents = documents,
    ids = ids,
    metadatas = metadatas
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 76.0MiB/s]


In [ ]:
print(f" Stored {len(documents)} chunks in chroma db")
print(f" Sources: 5 differnt files")

 Stored 120 chunks in chroma db
 Sources: 5 differnt files


# Building Retrievel Pipeline

In [ ]:
def retrieve(question, n_results = 3):
  results = collection.query(
      query_texts = [question],
      n_results = n_results
  )
  return results['documents'][0], results['metadatas'][0]

In [ ]:
chunks, sources = retrieve("What is the work from home policy?", 3)

for i in range(len(chunks)):
  print(f"--- Chunks {i+1} ---")
  print(f"Source: {sources[i]}")
  print(f"Text: {chunks[i]} ")
  print()

--- Chunks 1 ---
Source: {'source': 'HR Policy'}
Text: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval. 

--- Chunks 2 ---
Source: {'source': 'HR Policy'}
Text: NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026 

--- Chunks 3 ---
Source: {'source': 'HR Policy'}
Text: Regular WFH:
Employees may work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM IST) on WFH days. 



In [ ]:
def ask_rag(question, n_results = 3, verbose = True):
  chunks, sources = retrieve(question, n_results)

  if verbose:
    print(f"\n{'═' * 60}")
    print(f"❓ Question: {question}")
    print(f"{'─' * 60}")
    print(f"📄 Retrieved {len(chunks)} chunks:")
    for i, (chunk, source) in enumerate(zip(chunks, sources)):
        print(f"   [{source['source']}] {chunk[:80]}...")
    print(f"{'─' * 60}")


  context = "\n\n".join(chunks)

  messages= [
      {
          "role": "system",
          "content" : (
            "You are a helpful assistance that asnwers quesitosn based ONLY on the provided context"
            "If the context does not contain informaiton to asnwer this quesitons, "
            "say I dont have enough context or information to answer this question"
            "Don not make up information or assume anything. Strickly answer only from the providede context"
          )
      },
      {
          "role": "user",
          "content": f"Context: {context}\n\n---\n\nQuestion: {question}"
      }
  ]

  response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature = 0.2 # Low temperature for actual answers. not for creativity.
    )

  answer = response.choices[0].message.content

  if verbose:
    print(f" Answer: {answer}")
    print(f"{'═' * 60}")

  return answer

print("Rag Pipeline is built ")


Rag Pipeline is built 


# Test the RAG Pipeline

In [ ]:
ask_rag("What is Home Policy?")


════════════════════════════════════════════════════════════
❓ Question: What is Home Policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
   [Security] NovaTech Solutions — Information Security Policy
Classification: Internal | Vers...
   [Security] Company Laptops:
- Full disk encryption must be enabled (FileVault on Mac, BitLo...
────────────────────────────────────────────────────────────
 Answer: I don't have enough context or information to answer this question.
════════════════════════════════════════════════════════════


"I don't have enough context or information to answer this question."

In [ ]:
ask_rag("What is the work from Home Policy?")


════════════════════════════════════════════════════════════
❓ Question: What is the work from Home Policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
   [HR Policy] NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: J...
   [HR Policy] Regular WFH:
Employees may work from home up to 2 days per week. The preferred W...
────────────────────────────────────────────────────────────
 Answer: The Work From Home (WFH) policy at NovaTech Solutions allows all employees who have completed their probation period of 6 months to work from home. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval. Regular WFH arrangements permit employees to work from home up to 2 days per week, with preferred days being Wednesday and Friday, though adjustments can be made based on projec

'The Work From Home (WFH) policy at NovaTech Solutions allows all employees who have completed their probation period of 6 months to work from home. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval. Regular WFH arrangements permit employees to work from home up to 2 days per week, with preferred days being Wednesday and Friday, though adjustments can be made based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM IST) on WFH days.'

In [ ]:
ask_rag("How many days of annual leave do employees get?")


════════════════════════════════════════════════════════════
❓ Question: How many days of annual leave do employees get?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Annual Leave:
All full-time employees are entitled to 24 days of paid annual lea...
   [HR Policy] Sick Leave:
Employees are entitled to 12 days of sick leave per year. Sick leave...
   [HR Policy] Casual Leave:
Employees are entitled to 6 days of casual leave per year. Casual ...
────────────────────────────────────────────────────────────
 Answer: Employees are entitled to 24 days of paid annual leave per calendar year.
════════════════════════════════════════════════════════════


'Employees are entitled to 24 days of paid annual leave per calendar year.'

In [ ]:
ask_rag("What happens during the probation period?")


════════════════════════════════════════════════════════════
❓ Question: What happens during the probation period?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Probation Period:
All new employees undergo a probation period of 6 months from ...
   [HR Policy] Extension of Probation:
In cases where performance is borderline, probation may ...
   [HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
────────────────────────────────────────────────────────────
 Answer: During the probation period, all new employees undergo a 6-month evaluation period from the date of joining. Either party may terminate employment with 15 days notice or salary in lieu of notice. Additionally, if an employee's performance is borderline, the probation may be extended by up to 3 months with HR and department head approval, and this extension must be communicated in writing to the employee.
══════════════════════════

"During the probation period, all new employees undergo a 6-month evaluation period from the date of joining. Either party may terminate employment with 15 days notice or salary in lieu of notice. Additionally, if an employee's performance is borderline, the probation may be extended by up to 3 months with HR and department head approval, and this extension must be communicated in writing to the employee."

In [ ]:
# Product KB questions
ask_rag("What are the pricing plans for CloudDesk Pro?", 3)


════════════════════════════════════════════════════════════
❓ Question: What are the pricing plans for CloudDesk Pro?
────────────────────────────────────────────────────────────
📄 Retrieved 10 chunks:
   [Product] Support Channels:
- Help Center: docs.clouddesk.pro — Self-service articles and ...
   [Product] What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management and tea...
   [Product] CloudDesk Pro — Product Knowledge Base
Internal Support Reference | Version 2.1 ...
   [Product] Target Users:
CloudDesk Pro is designed for project managers, team leads, and in...
   [Product] Enterprise Plan — Custom pricing (contact sales):
- Everything in Business, plus...
   [Product] Starter Plan — Rs 299/user/month (billed annually) or Rs 399/user/month (billed ...
   [Product] Business Plan — Rs 699/user/month (billed annually) or Rs 899/user/month (billed...
   [Product] Annual Billing Discount:
All plans offer approximately 25% discount when billed ...
   [Product] Complia

'The pricing plans for CloudDesk Pro are as follows:\n\n1. **Starter Plan**: \n   - Rs 299/user/month (billed annually) or Rs 399/user/month (billed monthly)\n   - Up to 20 users\n   - 5 projects\n   - 10 GB storage\n   - Basic task management (Kanban only)\n   - Email support (response within 48 hours)\n   - No API access\n   - No custom workflows\n\n2. **Business Plan**: \n   - Rs 699/user/month (billed annually) or Rs 899/user/month (billed monthly)\n   - Unlimited users\n   - Unlimited projects\n   - 100 GB storage per user\n   - Full task management (Kanban + List + Gantt)\n   - Document Hub with version history\n   - Time Tracker\n   - Priority email and chat support (response within 4 hours)\n   - API access\n   - Custom workflows and automation\n   - SSO (Single Sign-On) support\n\n3. **Enterprise Plan**: \n   - Custom pricing (contact sales)\n   - Everything in Business, plus:\n     - Unlimited storage\n     - Dedicated account manager\n     - 24/7 phone support with 1-hour re

In [ ]:
ask_rag("How do I cancel my subscription?")


════════════════════════════════════════════════════════════
❓ Question: How do I cancel my subscription?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [Product] "How do I cancel my subscription?":
Go to Settings → Billing → Subscription → Ca...
   [Product] Refund Policy:
Monthly subscriptions: Full refund if cancelled within 48 hours o...
   [Product] Free Trial:
All new accounts start with a 14-day free trial of the Business plan...
────────────────────────────────────────────────────────────
 Answer: To cancel your subscription, go to Settings → Billing → Subscription → Cancel. You will be asked to confirm and provide a reason. Your account remains active until the end of the current billing period, and you can reactivate anytime before the period ends.
════════════════════════════════════════════════════════════


'To cancel your subscription, go to Settings → Billing → Subscription → Cancel. You will be asked to confirm and provide a reason. Your account remains active until the end of the current billing period, and you can reactivate anytime before the period ends.'

In [ ]:
ask_rag("What is the refund policy?")


════════════════════════════════════════════════════════════
❓ Question: What is the refund policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [Product] Refund Policy:
Monthly subscriptions: Full refund if cancelled within 48 hours o...
   [Product] Failed Payments:
If a payment fails, the system retries 3 times over 7 days. If ...
   [HR Policy] Reimbursement Process:
All expense claims must be submitted through the HR porta...
────────────────────────────────────────────────────────────
 Answer: The refund policy is as follows:

- Monthly subscriptions: Full refund if cancelled within 48 hours of charge. No refund after 48 hours.
- Annual subscriptions: Non-refundable. However, if cancelling within the first 30 days of a new annual subscription, a prorated refund (minus one month at monthly rates) may be issued upon request to support.
════════════════════════════════════════════════════════════


'The refund policy is as follows:\n\n- Monthly subscriptions: Full refund if cancelled within 48 hours of charge. No refund after 48 hours.\n- Annual subscriptions: Non-refundable. However, if cancelling within the first 30 days of a new annual subscription, a prorated refund (minus one month at monthly rates) may be issued upon request to support.'

In [ ]:
# A question that spans both documents — should say "not enough info"
ask_rag("What is the capital of France?")


════════════════════════════════════════════════════════════
❓ Question: What is the capital of France?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [Product] What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management and tea...
   [Onboarding] Communication:
- Slack: Primary daily communication. Check channels: #general, #...
   [HR Policy] Communication:
Slack is the primary communication tool for daily work. Email is ...
────────────────────────────────────────────────────────────
 Answer: I don't have enough context or information to answer this question.
════════════════════════════════════════════════════════════


"I don't have enough context or information to answer this question."

# Agentic RAG

# Create RAG Tool

In [ ]:
def search_docs(query:str) ->str:
  results = collection.query(
      query_texts = [query],
      n_results = 3
  )
  chunks = results['documents'][0]
  return "\n\n".join(chunks)

#Json Schema

rag_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search the company internal documents (HR Policy, Product, Security, onboarding, engineering) to find the relevant information. Use this when the user asks about company policies, products, features or support.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query you want to find relevant document"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

available_tools = {
    "search_docs": search_docs
}
print("Rag tool defined")

Rag tool defined


In [ ]:
def rag_agent(question, verbose= True):
    messages = [
        {
            "role":"system",
            "content": (
                "You are a helpful company assistance and you have access to internal documents "
                "including HR polices and Porduct, engineering , onboarding, security"
                "Use the search_docs tool to find answers from the company documents"
                "If the documents dont contian the answer, say so clearly."
                "Always base your answers on the rerieved documets if you decided to use the tool."
                "If tool use is not needed, specifcy that you are not using tool and asnwer the quesiton based on your knowledge."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    if verbose:
          print(f"\n{'═' * 60}")
          print(f"🧑 Question: {question}")
          print(f"{'─' * 60}")


    max_steps = 3
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=rag_tools,
        )

        choice = response.choices[0]

        # LLM is done
        if choice.finish_reason == "stop":
            if verbose:
                print(f"🤖 Answer: {choice.message.content}")
                print(f"{'═' * 60}")
            return choice.message.content

        # LLM wants to search
        if choice.message.tool_calls:
            messages.append(choice.message)

            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"  🔍 Searching: \"{args['query']}\"")

                result = available_tools[func_name](**args)

                if verbose:
                    print(f"  📄 Found {len(result)} chars of context")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

    return "Could not answer within step limit."

print("✅ RAG Agent ready!")

✅ RAG Agent ready!


In [ ]:
 # Test the RAG agent
rag_agent("How many sick leave days do I get per year?")


════════════════════════════════════════════════════════════
🧑 Question: How many sick leave days do I get per year?
────────────────────────────────────────────────────────────
  🔍 Searching: "sick leave days per year"
  📄 Found 961 chars of context
🤖 Answer: You are entitled to 12 days of sick leave per year. Please note that sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner, and sick leave cannot be carried forward or encashed.
════════════════════════════════════════════════════════════


'You are entitled to 12 days of sick leave per year. Please note that sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner, and sick leave cannot be carried forward or encashed.'

In [ ]:
rag_agent("What is 25*4 ?")


════════════════════════════════════════════════════════════
🧑 Question: What is 25*4 ?
────────────────────────────────────────────────────────────
🤖 Answer: The answer to 25 multiplied by 4 is 100.
════════════════════════════════════════════════════════════


'The answer to 25 multiplied by 4 is 100.'

In [ ]:
rag_agent("What is the capital of france?")


════════════════════════════════════════════════════════════
🧑 Question: What is the capital of france?
────────────────────────────────────────────────────────────
🤖 Answer: I am not using the tool for this question. The capital of France is Paris.
════════════════════════════════════════════════════════════


'I am not using the tool for this question. The capital of France is Paris.'

In [ ]:
# Try your own questions!
rag_agent("Can I carry forward unused annual leave?")


════════════════════════════════════════════════════════════
🧑 Question: Can I carry forward unused annual leave?
────────────────────────────────────────────────────────────
  🔍 Searching: "carry forward unused annual leave policy"
  📄 Found 961 chars of context
🤖 Answer: Yes, you can carry forward unused annual leave, but only up to 10 days to the next year. Any unused leave beyond 10 days will lapse on December 31st.
════════════════════════════════════════════════════════════


'Yes, you can carry forward unused annual leave, but only up to 10 days to the next year. Any unused leave beyond 10 days will lapse on December 31st.'